In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pprint import pprint

In [2]:
import sys
from pathlib import Path

# The saved models unpickle classes from `temporal_manifolds`; a minimal copy of
# that code lives in `vendor/` at the repo root.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vendor").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "vendor") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "vendor"))

from temporal_manifolds.viz.activation_pls import load_pls_model
from temporal_manifolds.viz.curve_fitting import load_curve_model

pls, pls_metadata = load_pls_model(
    REPO_ROOT / "models" / "ctype_only_activation_pls_layer_out-21.joblib"
)
spline, spline_metadata = load_curve_model(
    REPO_ROOT / "models" / "ctype_only_activation_curve_PLS1-PLS2-PLS3-by-t_spline.joblib"
)

print(pls, pls_metadata["component_count"], "components /", pls_metadata["feature_count"], "features")
print(spline.algorithm, spline.parameter_feature, "->", spline.coordinate_features)

PLSRegression(n_components=3) 3 components / 2560 features
spline t -> ('PLS1', 'PLS2', 'PLS3')


In [3]:
df = pd.read_csv("../data/squished_activation_pls_projection.csv")
df=df[df["source_folder"]=="<averaged>"]
fig = px.scatter_3d(df, x="PLS1", y="PLS2", z="PLS3", color="log10_time_horizon_months")
fig.update_traces(marker={"size":3})

# Overlay the fitted spline, sampled across its training range in t.
t_lo, t_hi = spline.training_parameter_bounds
t_grid = np.linspace(t_lo, t_hi, 400)
curve_xyz = spline.predict(t_grid)
fig.add_scatter3d(
    x=curve_xyz[:, 0],
    y=curve_xyz[:, 1],
    z=curve_xyz[:, 2],
    mode="lines",
    line={"width": 8, "color": "black"},
    name=f"{spline.algorithm} curve",
    customdata=t_grid,
    hovertemplate="t=%{customdata:.3f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<br>PLS3=%{z:.2f}<extra></extra>",
)
fig.update_layout(
    autosize=True,
    width=None,
    height=None
)
fig.show()

In [4]:
%pprint
pprint(spline_metadata)

Pretty printing has been turned OFF
{'aggregation_fields': ['time_horizon_months'],
 'artifact_kind': 'temporal_manifolds_curve_model',
 'artifact_version': 4,
 'cached_position': -1,
 'coordinate_features': ('PLS1', 'PLS2', 'PLS3'),
 'curve_data_sha256': '5671d76d1dc01b1aca9af13898d95239280073ccbae938b2e6ef46f6f6019719',
 'direction_method': 'PLS',
 'direction_target': 'log10_time_horizon_months',
 'fit_scope': 'Visible',
 'layer_component': 'layer_out/21',
 'parameter_feature': 't',
 'pca_fingerprint_version': 2,
 'pca_sha256': '6837a7651cc09c709bc46b9bda4db93f2ad59ca20a5666755dfd2c0e283c691d',
 'random_state': 42,
 'scipy_version': '1.17.1',
 'sklearn_version': '1.9.0'}


In [5]:
# Extrude the fitted spline along the PLS3 direction.
# Surface: S(t, u) = (PLS1(t), PLS2(t), u), i.e. the curve swept along PLS3,
# where the extrusion parameter u *is* the PLS3 coordinate.
u_lo, u_hi = df["PLS3"].min(), df["PLS3"].max()
pad = 0.05 * (u_hi - u_lo)
u_grid = np.linspace(u_lo - pad, u_hi + pad, 60)

T, U = np.meshgrid(t_grid, u_grid, indexing="ij")   # (n_t, n_u)
X = np.broadcast_to(curve_xyz[:, 0:1], T.shape)     # PLS1(t)
Y = np.broadcast_to(curve_xyz[:, 1:2], T.shape)     # PLS2(t)
Z = U                                               # PLS3 = u

fig.add_surface(
    x=X,
    y=Y,
    z=Z,
    surfacecolor=T,
    colorscale="Viridis",
    opacity=0.4,
    showscale=False,
    name="spline extruded along PLS3",
    showlegend=True,
    hovertemplate="t=%{surfacecolor:.3f}<br>u=PLS3=%{z:.2f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<extra></extra>",
)
fig.show()


In [ ]:
# Quadratic extrusion along PLS3, fitted to minimise geometric RMSE.
#
# Surface:  S(t, u) = ( PLS1(t) + p1(u),  PLS2(t) + p2(u),  u )
# with p_k(u) = c_k + a_k*u + b_k*u**2   (u *is* the PLS3 coordinate).
#
# Because the third coordinate of the surface is exactly u, a data point's
# closest surface point necessarily has u = PLS3 of that point.  The geometric
# distance therefore reduces to the in-plane (PLS1, PLS2) distance minimised
# over t, and we fit c, a, b by alternating least squares:
#   1. assign each point its nearest t on the current offset curve
#   2. refit the quadratic offsets in closed form given those t
P = df[["PLS1", "PLS2", "PLS3"]].to_numpy(float)
u_obs = P[:, 2]
V = np.column_stack([np.ones_like(u_obs), u_obs, u_obs**2])   # quadratic design

# Dense sampling of the base spline for the nearest-t search.
t_dense = np.linspace(t_lo, t_hi, 4000)
xy_dense = spline.predict(t_dense)[:, :2]

coef = np.zeros((3, 2))            # rows [c, a, b], cols [PLS1, PLS2]
for _ in range(50):
    offset = V @ coef                                   # (n, 2)
    target = P[:, :2] - offset                          # curve must match this
    d2 = ((target[:, None, :] - xy_dense[None, :, :]) ** 2).sum(-1)
    idx = d2.argmin(1)
    resid = P[:, :2] - xy_dense[idx]                    # explained by p(u)
    new_coef, *_ = np.linalg.lstsq(V, resid, rcond=None)
    if np.allclose(new_coef, coef, atol=1e-12):
        coef = new_coef
        break
    coef = new_coef

# Re-assign t under the final coefficients before scoring.
final_offset = V @ coef
idx = (((P[:, :2] - final_offset)[:, None, :] - xy_dense[None]) ** 2).sum(-1).argmin(1)
final_d = np.linalg.norm(P[:, :2] - (xy_dense[idx] + final_offset), axis=1)
rmse = np.sqrt((final_d**2).mean())

base_d = np.linalg.norm(
    P[:, :2] - xy_dense[
        ((P[:, None, :2] - xy_dense[None]) ** 2).sum(-1).argmin(1)
    ],
    axis=1,
)
print("geometric RMSE  straight extrusion:", np.sqrt((base_d**2).mean()).round(4))
print("geometric RMSE quadratic extrusion:", rmse.round(4))
print("offset coefficients [c, a, b] x [PLS1, PLS2]:\n", coef)

# Rebuild the surface with the fitted quadratic offsets.
Vg = np.column_stack([np.ones_like(u_grid), u_grid, u_grid**2])   # (n_u, 3)
off = Vg @ coef                                                   # (n_u, 2)
Xq = curve_xyz[:, 0:1] + off[None, :, 0]      # (n_t, n_u)
Yq = curve_xyz[:, 1:2] + off[None, :, 1]
Zq = np.broadcast_to(u_grid[None, :], Xq.shape)
Tq = np.broadcast_to(t_grid[:, None], Xq.shape)

fig_q = px.scatter_3d(df, x="PLS1", y="PLS2", z="PLS3", color="log10_time_horizon_months")
fig_q.update_traces(marker={"size": 3})
fig_q.add_surface(
    x=Xq, y=Yq, z=Zq,
    surfacecolor=Tq,
    colorscale="Viridis",
    opacity=0.45,
    showscale=False,
    name="quadratic extrusion (u = PLS3)",
    showlegend=True,
    hovertemplate="t=%{surfacecolor:.3f}<br>u=PLS3=%{z:.2f}<br>PLS1=%{x:.2f}<br>PLS2=%{y:.2f}<extra></extra>",
)
fig_q.update_layout(autosize=True, width=None, height=None)
fig_q.show()
